# TTISS On- and Off-Target Insertion Analysis

This notebook processes TTISS paired-end sequencing data to classify R2 retrotransposon insertion sites as on-target (25S rDNA), off-target, or plasmid-derived. Run cells in order. Complete Steps 1 and 2 in the README before starting — directory setup, bowtie2 index build, and barrnap 25S annotation.

## Configuration

Set `SAMPLE` to the prefix of the FASTQ files currently being processed (e.g. `rep1`, `rep2`). Set `NEG_CONTROL` to the prefix of the payload-only negative control. Set `SAMPLES` to the full list of replicates — this is used for figure generation after all samples have been processed.

In [ ]:
import os
import gzip
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

SAMPLE      = 'rep1'                  # sample currently being processed
NEG_CONTROL = 'neg_control'           # negative control prefix
SAMPLES     = ['rep1', 'rep2', 'rep3']  # all replicates, used for figure generation
K           = 30                      # flank length for insertion site extraction

os.environ['SAMPLE'] = SAMPLE
os.makedirs('final', exist_ok=True)

def count_lines(path):
    with open(path) as f:
        return sum(1 for _ in f)

## Step 3: Filter and trim reads

### Filter for R2Tg 3′ UTR

Retains only read pairs where R1 begins with the complete R2Tg 3′ UTR (`GGAACATATATAATTTATGTGTGTTCGATAAATAGC`), then removes the UTR from R1 to improve downstream mapping. Approximately 91% of R1 reads should pass this filter.

In [ ]:
%%bash
cutadapt -j 0 \
  -g 'GGAACATATATAATTTATGTGTGTTCGATAAATAGC;min_overlap=36;max_error_rate=0.05' \
  --no-indels \
  --discard-untrimmed \
  --pair-filter=first \
  --minimum-length 20 \
  -o final/${SAMPLE}_trim_R1.fastq.gz \
  -p final/${SAMPLE}_trim_R2.fastq.gz \
  data/${SAMPLE}/${SAMPLE}_R1.fastq.gz \
  data/${SAMPLE}/${SAMPLE}_R2.fastq.gz \
  > final/${SAMPLE}_cutadapt_1.log
cat final/${SAMPLE}_cutadapt_1.log

### Trim residual adaptor sequences

Trims the Tn5 ME motif from the 3′ end of R1 (`-a`) and its reverse complement from the 5′ end of R2 (`-G`), and removes the reverse complement of the R2Tg 3′ UTR from the 3′ end of R2 (`-A`). No reads are discarded in this step.

In [ ]:
%%bash
cutadapt -j 0 \
  -a 'CTGTCTCTTATACACATCT;min_overlap=12;max_error_rate=0.1' \
  -G 'AGATGTGTATAAGAGACAG;min_overlap=19;max_error_rate=0.1' \
  -A 'GCTATTTATCGAACACACATAAATTATATATGTTCC;min_overlap=15;max_error_rate=0.1' \
  --no-indels \
  --minimum-length 20 \
  -o final/${SAMPLE}_trim2_R1.fastq.gz \
  -p final/${SAMPLE}_trim2_R2.fastq.gz \
  final/${SAMPLE}_trim_R1.fastq.gz \
  final/${SAMPLE}_trim_R2.fastq.gz \
  > final/${SAMPLE}_cutadapt_2.log
cat final/${SAMPLE}_cutadapt_2.log

## Step 4: Align to genome and payload reference

Maps trimmed read pairs to the combined genome and payload index. Expect this to take 15–45 minutes per sample. The BAM is then sorted and indexed before classification.

In [ ]:
%%bash
bowtie2 -p 10 \
  -x index/benth_plasmid \
  -1 final/${SAMPLE}_trim2_R1.fastq.gz \
  -2 final/${SAMPLE}_trim2_R2.fastq.gz \
  -D 20 -R 3 -N 0 -L 30 -i S,1,0.50 \
  --no-unal \
  2> final/${SAMPLE}_bowtie.log \
  | samtools view -@ 4 -b -o final/${SAMPLE}_mapped.bam

samtools sort final/${SAMPLE}_mapped.bam -o final/${SAMPLE}_sorted.bam
samtools index -@ 10 final/${SAMPLE}_sorted.bam

cat final/${SAMPLE}_bowtie.log

## Step 5: Classify read pairs

All counting steps use the same flag filter (`-F 0x90C -f 0x1`), keeping only primary read pairs with both reads mapped.

### Count all mapped pairs

Extracts unique read names from all concordantly mapped pairs. This total is the denominator for the on/off-target rate calculation.

In [ ]:
%%bash
samtools view -F 0x90C -f 0x1 final/${SAMPLE}_sorted.bam \
  | cut -f1 | sort -u > final/${SAMPLE}_all_pairs_mapped.txt

echo "Total mapped pairs:"
wc -l < final/${SAMPLE}_all_pairs_mapped.txt

### Count plasmid-derived pairs

Both reads must map to the plasmid contig. Approximately 98% of mapped read pairs are expected here and are excluded from on/off-target analysis.

In [ ]:
%%bash
samtools view -F 0x90C -f 0x1 final/${SAMPLE}_sorted.bam plasmid \
  | cut -f1 \
  | sort \
  | uniq -c \
  | awk '$1==2 {print $2}' > final/${SAMPLE}_plasmid.txt

echo "Plasmid-derived pairs:"
wc -l < final/${SAMPLE}_plasmid.txt

### Count on-target pairs

Retains read pairs where both reads overlap a 25S rDNA site. Reads are first filtered to those overlapping the 25S feature BED, then an awk script confirms that both R1 (flag bit 0x40) and R2 (flag bit 0x80) of a pair are present.

In [ ]:
%%bash
samtools view -b -F 0x90C -f 0x1 final/${SAMPLE}_sorted.bam \
  | bedtools intersect -abam - -b genome/25S_features.bed -u \
  | samtools view - \
  | awk '{
      q=$1; f=$2+0
      if (int(f/64)%2)  a[q]=1
      if (int(f/128)%2) b[q]=1
    }
    END{
      for (q in a) if (b[q]) print q
    }' \
  | sort -u > final/${SAMPLE}_25S.txt

echo "On-target pairs (25S):"
wc -l < final/${SAMPLE}_25S.txt

### Count off-target pairs

Off-target insertions are all concordantly mapped pairs that do not map to the payload or a 25S site. Read names touching either category (by either member of the pair) are collected and subtracted from the full mapped set.

In [ ]:
%%bash
# Read names where either read touches the plasmid
samtools view -F 0x90C -f 0x1 final/${SAMPLE}_sorted.bam plasmid \
  | cut -f1 | sort -u > final/${SAMPLE}_pairs_touching_plasmid.txt

# Read names where either read touches a 25S site
samtools view -b -F 0x90C -f 0x1 final/${SAMPLE}_sorted.bam \
  | bedtools intersect -abam - -b genome/25S_features.bed -u \
  | samtools view - \
  | cut -f1 | sort -u > final/${SAMPLE}_pairs_touching_25S.txt

# Union of plasmid- and 25S-touching read names
cat final/${SAMPLE}_pairs_touching_plasmid.txt \
    final/${SAMPLE}_pairs_touching_25S.txt \
  | sort -u > final/${SAMPLE}_pairs_touching_plasmid_or_25S.txt

# Off-target = all mapped pairs minus plasmid/25S-touching pairs
comm -23 \
  final/${SAMPLE}_all_pairs_mapped.txt \
  final/${SAMPLE}_pairs_touching_plasmid_or_25S.txt \
  > final/${SAMPLE}_pairs_other.txt

echo "Off-target pairs:"
wc -l < final/${SAMPLE}_pairs_other.txt

## Negative control correction

The payload-only negative control (no R2 protein) is processed through the full pipeline. Any on- or off-target reads in that sample represent PCR chimeras rather than true integration events. The false-positive rates from the negative control are applied to correct the experimental counts.

Set `NEG_CONTROL` in the configuration cell above and ensure the negative control has been processed through Steps 3–5 before running this cell.

In [ ]:
# Negative control counts
nc_all = count_lines(f'final/{NEG_CONTROL}_all_pairs_mapped.txt')
nc_on  = count_lines(f'final/{NEG_CONTROL}_25S.txt')
nc_off = count_lines(f'final/{NEG_CONTROL}_pairs_other.txt')

fp_on  = nc_on  / nc_all
fp_off = nc_off / nc_all

# Sample counts
s_all = count_lines(f'final/{SAMPLE}_all_pairs_mapped.txt')
s_on  = count_lines(f'final/{SAMPLE}_25S.txt')
s_off = count_lines(f'final/{SAMPLE}_pairs_other.txt')

# Corrected counts
corr_on  = s_on  - (fp_on  * s_all)
corr_off = s_off - (fp_off * s_all)
on_target_rate = corr_on / (corr_on + corr_off)

print(f'Negative control ({NEG_CONTROL})')
print(f'  Total mapped pairs : {nc_all:,}')
print(f'  On-target  (raw)   : {nc_on:,}  ->  FP rate = {fp_on:.4f}')
print(f'  Off-target (raw)   : {nc_off:,}  ->  FP rate = {fp_off:.4f}')
print()
print(f'Sample ({SAMPLE})')
print(f'  Total mapped pairs  : {s_all:,}')
print(f'  On-target  observed : {s_on:,}')
print(f'  Off-target observed : {s_off:,}')
print(f'  On-target  corrected: {corr_on:.0f}')
print(f'  Off-target corrected: {corr_off:.0f}')
print()
print(f'On-target rate: {on_target_rate:.3f}  ({on_target_rate * 100:.1f}%)')

## Step 6: Extract insertion flanks

For each read pair classified as on- or off-target, extracts `K` bp of genomic reference sequence on either side of the insertion junction. Run once per sample after Steps 3–5. Output is saved to `final/{SAMPLE}_insertion_flanks_k30.tsv` and used for sequence logo generation.

In [ ]:
import pysam

SHIFT = 0  # junction coordinate shift

def rc(seq):
    comp = str.maketrans('ACGTNacgtn', 'TGCANtgcan')
    return seq.translate(comp)[::-1]

def open_maybe_gz(path):
    return gzip.open(path, 'rt') if path.endswith('.gz') else open(path, 'rt')

def load_id_set(path):
    with open(path) as f:
        return {line.strip() for line in f if line.strip()}

def read_trimmed_r1_first_k(fastq_path, want_ids, k):
    out = {}
    with open_maybe_gz(fastq_path) as fh:
        while True:
            h = fh.readline()
            if not h:
                break
            seq = fh.readline().rstrip('\n')
            fh.readline()  # +
            fh.readline()  # qual
            rid = h.strip().lstrip('@').split()[0]
            if rid in want_ids:
                out[rid] = seq[:k]
    return out

In [ ]:
bam_path    = f'final/{SAMPLE}_sorted.bam'
fasta_path  = 'genome/genome_plasmid.fasta'
out_tsv     = f'final/{SAMPLE}_insertion_flanks_k{K}.tsv'

on_ids  = load_id_set(f'final/{SAMPLE}_25S.txt')
off_ids = load_id_set(f'final/{SAMPLE}_pairs_other.txt')
cls     = {rid: 'on_target'  for rid in on_ids}
cls.update({rid: 'off_target' for rid in off_ids})
want_ids = set(cls)
print(f'on: {len(on_ids):,}  off: {len(off_ids):,}')

bam  = pysam.AlignmentFile(bam_path, 'rb')
fa   = pysam.FastaFile(fasta_path)
rows, seen = [], set()

for a in bam.fetch(until_eof=True):
    rid = a.query_name
    if rid not in want_ids or rid in seen or not a.is_read1:
        continue
    if a.is_unmapped or a.is_secondary or a.is_supplementary:
        continue

    contig     = a.reference_name
    contig_len = fa.get_reference_length(contig)

    if not a.is_reverse:
        jct        = max(0, a.reference_start - SHIFT)
        ref_before = fa.fetch(contig, max(0, jct - K), jct)
        ref_after  = fa.fetch(contig, jct, min(contig_len, jct + K))
    else:
        jct        = min(contig_len, a.reference_end + SHIFT)
        ref_after  = rc(fa.fetch(contig, max(0, jct - K), jct))
        ref_before = rc(fa.fetch(contig, jct, min(contig_len, jct + K)))

    rows.append({
        'read_id':         rid,
        'class':           cls[rid],
        'contig':          contig,
        'junction_1based': jct + 1,
        'strand':          '-' if a.is_reverse else '+',
        'mapq':            a.mapping_quality,
        'ref_before_k':    ref_before,
        'ref_after_k':     ref_after,
    })
    seen.add(rid)

bam.close()
fa.close()

flanks = pd.DataFrame(rows)
flanks.to_csv(out_tsv, sep='\t', index=False)
print(f'Wrote {len(flanks):,} rows to {out_tsv}')
print(flanks['class'].value_counts())

---
## Figures

Run the cells below after all samples in `SAMPLES` have been processed through Steps 3–6. Each figure reads directly from the pipeline output files and saves a PDF to the working directory.

### Figure 1: On/off-target insertion rate

Stacked bar showing the proportion of mapped read pairs classified as on-target (25S rDNA) or off-target, pooled across all replicates.

In [ ]:
mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 9,
    'axes.labelsize': 11, 'xtick.labelsize': 11, 'ytick.labelsize': 9,
})

on_counts  = [count_lines(f'final/{s}_25S.txt')         for s in SAMPLES]
off_counts = [count_lines(f'final/{s}_pairs_other.txt') for s in SAMPLES]

on_total  = sum(on_counts)
off_total = sum(off_counts)
total     = on_total + off_total
on_pct    = on_total  / total * 100
off_pct   = off_total / total * 100

fig, ax = plt.subplots(figsize=(2, 4))
ax.bar([0], [on_pct],  width=1, label='On-target',  color='whitesmoke', edgecolor='black')
ax.bar([0], [off_pct], width=1, label='Off-target', color='dimgrey',    edgecolor='black', bottom=[on_pct])

ax.set_ylabel('Sequencing Reads (%)')
ax.set_yticks(range(0, 101, 20))
ax.set_ylim(0, 104)
ax.set_xlim(-1, 1)
ax.set_xticks([])
ax.tick_params(axis='y', which='both', direction='in')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('fig_1.pdf')
print(f'On-target:  {on_total:,} ({on_pct:.2f}%)')
print(f'Off-target: {off_total:,} ({off_pct:.2f}%)')

### Figure 2: Sequence logo at insertion sites

Sequence information logo centered on the insertion junction, built from genomic reference flanks extracted in Step 6. Plotted separately for on-target and off-target insertions, pooled across all replicates. X-axis labels show the 25S rDNA sequence at the insertion site.

In [ ]:
import logomaker

BASES = ['A', 'C', 'G', 'T']

def pwm_from_seqs(seqs, alphabet=BASES):
    seqs = [s for s in seqs if isinstance(s, str) and len(s) > 0]
    L = len(seqs[0])
    counts = {b: np.ones(L) for b in alphabet}  # Laplace pseudocounts
    for s in seqs:
        for i, ch in enumerate(s):
            if ch in counts:
                counts[ch][i] += 1.0
    mat = pd.DataFrame({b: counts[b] for b in alphabet})
    return mat.div(mat.sum(axis=1), axis=0)

def info_logo_matrix(seqs, alphabet=BASES):
    pwm  = pwm_from_seqs(seqs, alphabet)
    H    = -(pwm * np.log2(pwm)).sum(axis=1)        # Shannon entropy (bits)
    Rseq = np.log2(len(alphabet)) - H               # information content
    heights = pwm.copy()
    for b in alphabet:
        heights[b] = pwm[b] * Rseq
    return heights

def plot_logo(seqs, k_before, k_after, trim_right=0, seq_labels=None, out_pdf='logo.pdf'):
    if trim_right:
        seqs    = [s[:-trim_right] for s in seqs]
        k_after -= trim_right

    heights    = info_logo_matrix(seqs)
    pos_labels = list(range(-k_before, 0)) + list(range(1, k_after + 1))
    heights.index = pos_labels

    color_scheme = {
        'A': '#FCD163', 'C': '#6FB0DE', 'G': '#5D72E8', 'T': '#FB8072', 'N': '#BDBDBD'
    }
    mpl.rcParams.update({
        'font.family': 'Arial', 'font.size': 9,
        'axes.labelsize': 11, 'xtick.labelsize': 11, 'ytick.labelsize': 9,
    })

    fig, ax = plt.subplots(figsize=(10, 3))
    logomaker.Logo(heights, ax=ax, color_scheme=color_scheme)
    ax.axvline(x=0, linestyle='--', color='black', linewidth=1)

    if seq_labels is not None:
        ax.set_xticks(pos_labels)
        ax.set_xticklabels(list(seq_labels), fontsize=8)
    else:
        ticks = pos_labels[::2]
        ax.set_xticks(ticks)
        ax.set_xticklabels(ticks)
        ax.set_xlabel('Position relative to insertion junction')

    ax.set_ylabel('Bits')
    ax.set_yticks([0, 0.2, 0.4, 0.6])
    ax.set_ylim(0, 0.65)
    ax.tick_params(axis='y', direction='in')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(out_pdf)
    print(f'Saved {out_pdf}')

In [ ]:
K_BEFORE   = 26
K_AFTER    = 19
MOVE_N     = 4
TRIM_RIGHT = 9
SEQ_LABELS = 'CGGGAGTAACTATGACTCTCTTAAGGTAGCCAAATG'  # 25S rDNA sequence at insertion site

paths = [f'final/{s}_insertion_flanks_k{K}.tsv' for s in SAMPLES]
df    = pd.concat([pd.read_csv(p, sep='\t') for p in paths], ignore_index=True)

# Shift window 4 bp toward the insert to centre the insertion site motif
before     = df['ref_before_k'].astype(str).str[-K:]
after      = df['ref_after_k'].astype(str)
moved      = before.str[-MOVE_N:]
before_adj = before.str[:-MOVE_N]                          # 26 bp
after_adj  = (moved + after).str[:(K_AFTER + MOVE_N)]      # 19 bp
df['ref_window'] = before_adj + after_adj                   # 45 bp

off_seqs = df.loc[df['class'] == 'off_target', 'ref_window'].tolist()

print(f'off-target sequences: {len(off_seqs):,}')
print(f'window length: {len(df["ref_window"].iloc[0])} bp')

In [ ]:
plot_logo(
    off_seqs,
    k_before=K_BEFORE, k_after=K_AFTER, trim_right=TRIM_RIGHT,
    seq_labels=SEQ_LABELS,
    out_pdf='fig_2.pdf',
)